In [ ]:
"""
CS 499 Computer Science Capstone!
Category Three: Databases

Artifact: Animal Shelter Dashboard
Original Course: CS 340 Client/Server Development
Original Creation Date: December 2025
Enhancements Date: July 27 - Aug 2, 2026
Student: Danny Morse

Overview:
This dashboard displays animal shelter records stored in the MongoDB. It allows
a user to look at records with a table, filter animals by category,
compare breeds, and then display an animal's location on a map

Enhancement Summary:
The original dashboard functionality was retained and improved in several
main areas:
    1. database reliability
    2. data visualization
    3. error handling
    4. maintainability

The enhanced dashboard now:
    - connects through the updated CRUD module
    - supports environment variables and a separate testing collection
    - checks the MongoDB connection before loading the dashboard
    - displays clear messages when chart or map data is unavailable
    - limits the breed chart to the ten most common breeds so it does't look to cluttered
    - uses a bar chart to compare breeds
    - checks for invalid map coordinates
    - displays the selected animal's name and breed on the map
    - keeps the breed chart and location map organized in a side by side viewer

Testing:
The enhanced dashboard was tested locally using MongoDB Compass, Visual
Studio Code, and a separate animals_test collection. Sample animal records
were used to verify the rescue filters, data table, breed chart, selected
row behavior, and map location features.

Outcome Alignment:
This enhancement demonstrates progress toward creating a DB driven
solution that is more efficient with quality of life improvements. Additionally 
organizational decision making is touched on by presenting animal 
records through filters, visual comparisons, and visual location info.
The added connection checks, imrpoved data handling, and validation also demonstrates 
progress toward using more secure and dependable practices.
"""

# Setup the Jupyter version of Dash
from dash import Dash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################

# Uses the original animals collection by default.
# The settings can still be redirected through environment variables for testing.
os.environ.setdefault("AAC_HOST", "localhost")
os.environ.setdefault("AAC_PORT", "27017")
os.environ.setdefault("AAC_DATABASE", "aac")
os.environ.setdefault("AAC_COLLECTION", "animals")

# Connect to MongoDB using the enhanced CRUD module.
db = AnimalShelter()

# Stops the dashboard if the database cannot be reached.
if not db.check_connection():
    raise ConnectionError(
        "The dashboard could not connect to MongoDB, please make sure the MongoDB service is running!"
    )

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB adds an id field that the Dash cannot display correctly, this will prevent issues if the field is missing
df.drop(columns=["_id"], errors="ignore", inplace=True)



## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = Dash(__name__)

image_filename = 'Grazioso Salvare Logo.png' 
encoded_image = base64.b64encode(open(image_filename, 'rb').read())



app.layout = html.Div([
#    html.Div(id='hidden-div', style={'display':'none'}),
    html.Center(html.B(html.H1('CS-340 Dashboard'))),
    
    html.Center(
        html.A(
            html.Img(
                src='data:image/png;base64,{}'.format(encoded_image.decode()),
                style={'height': '120px'}
            ),
            href='https://www.snhu.edu',
            target='_blank'
        )
    ),

    html.Center(html.H4("Created by Danny Morse")),

    html.Hr(),

   
 
    html.Div([
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'WATER'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'WILDERNESS'},
                {'label': 'Disaster or Individual Tracking', 'value': 'DISASTER'},
                {'label': 'Reset', 'value': 'RESET'}
            ],
            value='RESET',
            inline=True
        )
    ]),


dash_table.DataTable(
    id='datatable-id',
    columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
    data=df.to_dict('records'),

    # Interactive features 
    page_action='native',
    page_size=12,
    sort_action='native',
    filter_action='native',

    # make existing callbacks happy
    row_selectable='single',
    selected_rows=[],              
    column_selectable='single',    
    selected_columns=[],           

    style_table={'overflowX': 'auto'},
    style_cell={'textAlign': 'left', 'padding': '8px'},
    style_header={'fontWeight': 'bold'}
),



                        
    html.Br(),
    html.Hr(),

    # Places the breed chart and animal map beside each other for ease of access
        style={
            "display": "flex",
            "width": "100%",
            "gap": "10px"
        },
        children=[
            html.Div(
                id="graph-id",
                style={
                    "width": "50%",
                    "padding": "10px"
                }
            ),
            html.Div(
                id="map-id",
                style={
                    "width": "50%",
                    "padding": "10px"
                }
            )
        ]
    )
])

#############################################
# Interaction Between Components / Controller
#############################################



    
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
    if filter_type == "WATER":
        query = {
            "animal_type": "Dog",
            "breed": {"$regex": "Labrador Retriever|Chesapeake Bay Retriever|Newfoundland", "$options": "i"},
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    elif filter_type == "WILDERNESS":
        query = {
            "animal_type": "Dog",
            "breed": {"$regex": "German Shepherd|Alaskan Malamute|Old English Sheepdog|Siberian Husky|Rottweiler", "$options": "i"},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    elif filter_type == "DISASTER":
        query = {
            "animal_type": "Dog",
            "breed": {"$regex": "Doberman Pinscher|German Shepherd|Golden Retriever|Bloodhound|Rottweiler", "$options": "i"},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
        }

    else:
        query = {}

    dff = pd.DataFrame.from_records(db.read(query))

    if "_id" in dff.columns:
        dff.drop(columns=["_id"], inplace=True)

    return dff.to_dict("records")



# Displays the ten most common breeds in the current table results
@app.callback(
    Output("graph-id", "children"),
    [Input("datatable-id", "derived_virtual_data")]
)
def update_graphs(view_data):
    if not view_data:
        return html.P("No animal records are available for the chart.")

    dff = pd.DataFrame(view_data)

    if "breed" not in dff.columns or dff["breed"].dropna().empty:
        return html.P("Breed information is not available.")

    # Limits the chart to the ten most common breeds so it stays readable and uncluttered
    breed_counts = (
        dff["breed"]
        .dropna()
        .value_counts()
        .head(10)
        .reset_index()
    )

    breed_counts.columns = ["breed", "count"]

    fig = px.bar(
        breed_counts,
        x="breed",
        y="count",
        title="Top Animal Breeds",
        labels={
            "breed": "Breed",
            "count": "Number of Animals"
        }
    )

    fig.update_layout(xaxis_tickangle=-35)

    return [dcc.Graph(figure=fig)]

    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if not selected_columns:
        return []
    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]



"""
Updates the map when an animal is selected from the table
The table data is provided as a list of dictionaries
The selected row number is used to locate the matching record
Named DB fields are used instead of fixed column positions
"""

@app.callback(
    Output("map-id", "children"),
    [
        Input("datatable-id", "derived_virtual_data"),
        Input("datatable-id", "selected_rows")
    ]
)
def update_map(view_data, selected_rows):
    # Doesn't build a marker until the user selects an animal
    if not view_data or not selected_rows:
        return html.P("Select an animal from the table to view its location.")

    dff = pd.DataFrame(view_data)
    selected_row = selected_rows[0]

    # Makes sure the selected row still exists 
    if selected_row >= len(dff):
        return html.P("The selected animal could not be found.")

    animal = dff.iloc[selected_row]

    # These fields are needed to place and label the marker.
    required_columns = {
        "location_lat",
        "location_long",
        "breed",
        "name"
    }

    if not required_columns.issubset(dff.columns):
        return html.P(
            "The location information needed for the map is missing."
        )

    # Convert the coordinates to numbers in case they were stored as text first
    latitude = pd.to_numeric(
        animal["location_lat"],
        errors="coerce"
    )

    longitude = pd.to_numeric(
        animal["location_long"],
        errors="coerce"
    )

    if pd.isna(latitude) or pd.isna(longitude):
        return html.P(
            "The selected animal doesn't have valid coordinates"
        )

    animal_name = animal.get("name", "Unknown")
    animal_breed = animal.get("breed", "Unknown breed")

    return [
        dl.Map(
            style={
                "width": "100%",
                "height": "500px"
            },
            center=[latitude, longitude],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),

                dl.Marker(
                    position=[latitude, longitude],
                    children=[
                        dl.Tooltip(str(animal_breed)),

                        dl.Popup([
                            html.H4("Animal Information"),
                            html.P(f"Name: {animal_name}"),
                            html.P(f"Breed: {animal_breed}")
                        ])
                    ]
                )
            ]
        )
    ]
app.run(jupyter_mode="inline", debug=False) 

INFO: MongoDB connection was successful!
